# S08 · Your first full project — a housing-price predictor

We build a complete, honest house-price model, the kind you could hand to a
stakeholder. We load real data, lock away a test set, do all the prep and the model
inside one tidy object, let cross-validation choose the penalty knob, and judge the
model on flats it has never seen.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press play on each cell,
  top to bottom, and read the plain-English note above each one.
- New to the idea of a model over-trusting the data? Open
  `primers/overfitting_and_regularization.md` first.
- Already confident with code or with scikit-learn pipelines? Skip to the cell
  marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, pandas, matplotlib and scikit-learn.
# Google Colab already ships all four, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                                           # fast maths on numbers
import pandas as pd                                           # tables of data
import matplotlib.pyplot as plt                               # drawing charts
from sklearn.datasets import fetch_california_housing         # a real housing dataset
from sklearn.datasets import make_regression                  # backup data if offline
from sklearn.model_selection import train_test_split          # splitting off a test set
from sklearn.model_selection import GridSearchCV              # trying several alpha values
from sklearn.pipeline import Pipeline                         # bundle prep + model together
from sklearn.preprocessing import StandardScaler              # put features on one scale
from sklearn.linear_model import Ridge                        # the regularised model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set a seed so everyone gets the same split and the same results.
np.random.seed(0)

## Step 1 — load real housing data

We use the California housing dataset that ships with scikit-learn. Each row is a
neighbourhood, and the target is its median house value. The first time this runs
it fetches the data over the internet, so if there is no connection we quietly fall
back to a made-up dataset. Either way, the notebook always runs.

In [ ]:
try:
    data = fetch_california_housing(as_frame=True)
    features = data.data        # the inputs: a table of numbers about each area
    target = data.target        # what we predict: the median house value
    print("Loaded California housing:", features.shape[0], "rows,",
          features.shape[1], "columns")
except Exception:
    # Offline fallback: build a simple synthetic table so the notebook still runs.
    print("Could not download the housing data; using a synthetic dataset instead.")
    synthetic_inputs, synthetic_output = make_regression(
        n_samples=2000, n_features=8, noise=20.0, random_state=0)
    column_names = []
    for column_number in range(8):
        column_names.append("feature_" + str(column_number))
    features = pd.DataFrame(synthetic_inputs, columns=column_names)
    target = pd.Series(synthetic_output, name="target")
    print("Synthetic data ready:", features.shape[0], "rows,",
          features.shape[1], "columns")

# Show the first few rows of the input table.
print(features.head())

## Step 2 — split off a test set and lock it away

We train on one part of the data and lock the other part in a drawer. We open that
drawer only once, at the very end, to get an honest score. This is the single most
important habit in machine learning: never let the model see its final exam in
advance.

In [ ]:
# 80% for training, 20% kept aside for the final honest test.
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=0)

print("training flats:", features_train.shape[0])
print("test flats    :", features_test.shape[0])

## Step 3 — why we must put features on one scale

The penalty charges a fee for large weights and treats every feature the same. But
our features live on wildly different scales: income might be in the tens while a
count of rooms is a small handful. Without a fix, the penalty would punish those
features unfairly, just because of their units. The fix is to **standardise** every
feature: shift and stretch it so they all sit on a comparable footing. Look at how
different the raw ranges are first.

In [ ]:
# The spread (largest minus smallest value) differs a lot between columns.
print("Range of each feature (max minus min), before scaling:")
print((features_train.max() - features_train.min()).round(2))

## Step 4 — build a Pipeline: scaler + Ridge as one tidy object

A **Pipeline** is one object that does all the prep and the model together, in
order. Here it is two steps: first `StandardScaler` (does the scaling), then `Ridge`
(the model). Bundling them matters for one big reason: it stops **leakage**. When
the scaling happens *inside* the pipeline, the scaler learns its shift-and-stretch
numbers only from the training flats, never from the locked-away test flats. If we
scaled the whole dataset before splitting, information from the test set would leak
into training and our score would look better than it really is.

In [ ]:
# A pipeline is a list of (name, step) pairs, run in order.
housing_pipeline = Pipeline([
    ("scaler", StandardScaler()),    # step 1: put every feature on one scale
    ("model", Ridge(alpha=1.0)),     # step 2: the regularised model
])

# Fit the WHOLE pipeline on the training flats only.
# The scaler learns from the training flats, then Ridge is fitted on the scaled data.
housing_pipeline.fit(features_train, target_train)

print("Pipeline fitted: it scales, then runs Ridge, as one object.")

## Step 5 — let cross-validation choose alpha

We do not want to guess the `alpha` knob by hand. **Cross-validation** does it for
us. It splits the training data into folds, trains on some folds and checks on the
rest, then averages the score, and it repeats this for each `alpha` we offer. We use
`GridSearchCV` to try a grid of values and keep the best. Because the scaler lives
inside the pipeline, it is re-fitted fresh in every fold, so there is still no
leakage.

In [ ]:
# The alpha values to try. The name "model__alpha" means:
# the "alpha" setting of the step called "model" inside the pipeline.
alpha_grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}

# 5-fold cross-validation, scored by (negative) mean squared error.
grid_search = GridSearchCV(
    housing_pipeline,
    param_grid=alpha_grid,
    cv=5,
    scoring="neg_mean_squared_error")

# Run the whole search on the training flats only.
grid_search.fit(features_train, target_train)

print("Best alpha found:", grid_search.best_params_["model__alpha"])

## Step 6 — look at the cross-validation scores

Let us see the average error for each `alpha` we tried. The curve is usually
U-shaped: too small an `alpha` overfits, too large an `alpha` makes the model too
timid, and the best is somewhere in the middle. That dip is the sweet spot
cross-validation just found for us.

In [ ]:
# GridSearchCV stores the mean score for each alpha it tried.
# The scores are negative MSE (higher is better), so we flip the sign to get MSE.
tried_alphas = alpha_grid["model__alpha"]
mean_errors = -grid_search.cv_results_["mean_test_score"]

print("Cross-validation mean squared error for each alpha:")
for one_alpha, one_error in zip(tried_alphas, mean_errors):
    print("  alpha =", one_alpha, " -> MSE =", round(one_error, 4))

plt.figure(figsize=(8, 5))
plt.plot(tried_alphas, mean_errors, marker="o", color="#2E75B6")
plt.xscale("log")
plt.xlabel("alpha (penalty strength, log scale)")
plt.ylabel("cross-validated mean squared error")
plt.title("Pick the alpha at the bottom of the curve")
plt.show()

## Step 7 — open the drawer: score on the hidden test set

Now, and only now, we unlock the test set. `grid_search` already kept the best
pipeline (the winning `alpha`, re-fitted on all the training flats). We ask it to
predict the test prices and report three honest numbers. These are the numbers you
would put in front of a stakeholder.

In [ ]:
# grid_search.best_estimator_ is the best pipeline, ready to predict.
best_model = grid_search.best_estimator_

# Predict prices for the flats the model has never seen.
test_predictions = best_model.predict(features_test)

mse = mean_squared_error(target_test, test_predictions)
mae = mean_absolute_error(target_test, test_predictions)
r2 = r2_score(target_test, test_predictions)

print("Test-set results (data the model never saw):")
print("  MSE       :", round(mse, 4), " (average squared error - punishes big misses)")
print("  MAE       :", round(mae, 4), " (average size of the miss)")
print("  R squared :", round(r2, 4), " (share of the variation explained; 1.0 is perfect)")

## Step 8 — a picture: predicted vs actual

If the model were perfect, every point would sit on the diagonal line. The closer
the cloud hugs the line, the better the model. Points far from the line are the
flats the model got most wrong.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(target_test, test_predictions, alpha=0.3, color="#2E75B6")

# The perfect-prediction diagonal line, for comparison.
lowest_value = min(target_test.min(), test_predictions.min())
highest_value = max(target_test.max(), test_predictions.max())
plt.plot([lowest_value, highest_value], [lowest_value, highest_value],
         color="#C0392B", linewidth=2, label="perfect prediction")

plt.xlabel("actual value")
plt.ylabel("predicted value")
plt.title("Predicted vs actual on the test set (closer to the line is better)")
plt.legend()
plt.show()

## Step 9 — read the model back in plain language

Because every feature was standardised, the weights are now directly comparable: a
bigger weight means that feature pushes the price harder. We reach inside the
pipeline, grab the fitted Ridge step, and read its weights. This is how you turn a
model into a sentence a non-technical stakeholder understands.

In [ ]:
# Reach into the pipeline and grab the fitted Ridge step.
fitted_ridge = best_model.named_steps["model"]
learned_coefficients = fitted_ridge.coef_

# Pair each feature name with its weight.
feature_names = list(features.columns)

print("Weight for each feature (on standardised features):")
for name, coefficient in zip(feature_names, learned_coefficients):
    print("  ", name, ":", round(coefficient, 3))

print("\nA large positive number means that feature pushes the predicted price up;")
print("a large negative number pushes it down.")

### Stretch (optional) — a ColumnTransformer for mixed data

Skip this if you are new to code. Real datasets are often not all numbers: a flat
also has text columns like the locality name or "furnished / unfurnished". Numbers
need scaling; text categories need to be turned into 0/1 columns instead. A
`ColumnTransformer` lets one pipeline apply a different prep to each kind of column.

The California data is all numeric, so we did not need this above. Here is the
pattern you would use the moment a categorical column appears, so it is ready when
you meet one.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Pretend our data had these two kinds of columns.
numeric_columns = list(features.columns)     # scale these
categorical_columns = []                     # would list text columns here

# Tell the ColumnTransformer which prep to apply to which columns.
preprocess = ColumnTransformer([
    ("scale_numbers", StandardScaler(), numeric_columns),
    ("encode_text", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
])

# Drop it into a pipeline exactly like the StandardScaler before.
mixed_pipeline = Pipeline([
    ("prep", preprocess),
    ("model", Ridge(alpha=grid_search.best_params_["model__alpha"])),
])
mixed_pipeline.fit(features_train, target_train)

mixed_r2 = r2_score(target_test, mixed_pipeline.predict(features_test))
print("Same result via a ColumnTransformer, R squared:", round(mixed_r2, 4))
print("With no categorical columns it matches Step 7; add text columns and it just works.")

## What you just did

You ran a complete, honest machine-learning project, the arc you will repeat for
the rest of the course:

1. Loaded real data and locked away a test set right away.
2. Put the scaling and the model in **one Pipeline**, so scaling happened on
   training flats only. No leakage.
3. Chose the penalty knob `alpha` by **cross-validation**, never by peeking at the
   test set.
4. Reported MSE, MAE and R² on the hidden test set, and looked at the
   predicted-vs-actual plot.
5. Read the standardised weights to explain, in plain words, what drives the price.

This is your first full project deliverable. Save the notebook, write a short,
honest paragraph on what the model does well and where it struggles, and that is
something you could hand to a stakeholder or put in a portfolio.